# Cover type classification using NN
In this project I trained a custom NN to classify cover type. I used optuna to find best parameters for this task.

In [1]:
import torch
from utils import load_device

In [2]:
device=load_device()

PyTorch Version: 2.8.0+cu126
Using GPU: NVIDIA GeForce GTX 1660 Ti


In [3]:
from sklearn.datasets import fetch_covtype
cov=fetch_covtype()
X,y=cov.data,cov.target

In [4]:
import pandas as pd 
df=pd.DataFrame(X,columns=cov.feature_names)
df.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39
0,2596.0,51.0,3.0,258.0,0.0,510.0,221.0,232.0,148.0,6279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2590.0,56.0,2.0,212.0,-6.0,390.0,220.0,235.0,151.0,6225.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2804.0,139.0,9.0,268.0,65.0,3180.0,234.0,238.0,135.0,6121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2785.0,155.0,18.0,242.0,118.0,3090.0,238.0,238.0,122.0,6211.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2595.0,45.0,2.0,153.0,-1.0,391.0,220.0,234.0,150.0,6172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df.isnull().sum(axis=0)

Elevation                             0
Aspect                                0
Slope                                 0
Horizontal_Distance_To_Hydrology      0
Vertical_Distance_To_Hydrology        0
Horizontal_Distance_To_Roadways       0
Hillshade_9am                         0
Hillshade_Noon                        0
Hillshade_3pm                         0
Horizontal_Distance_To_Fire_Points    0
Wilderness_Area_0                     0
Wilderness_Area_1                     0
Wilderness_Area_2                     0
Wilderness_Area_3                     0
Soil_Type_0                           0
Soil_Type_1                           0
Soil_Type_2                           0
Soil_Type_3                           0
Soil_Type_4                           0
Soil_Type_5                           0
Soil_Type_6                           0
Soil_Type_7                           0
Soil_Type_8                           0
Soil_Type_9                           0
Soil_Type_10                          0


In [6]:
import numpy as np
n_classes=len(np.unique(y))
n_features=X.shape[1]

In [7]:
np.unique_counts(y)

UniqueCountsResult(values=array([1, 2, 3, 4, 5, 6, 7], dtype=int32), counts=array([211840, 283301,  35754,   2747,   9493,  17367,  20510]))

In [8]:
n_features

54

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,random_state=42,shuffle=True,train_size=0.95)

In [10]:
X_dev,X_test,y_dev,y_test=train_test_split(X_test,y_test,stratify=y_test,random_state=42,train_size=0.5)

In [11]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_dev=scaler.transform(X_dev)
X_test=scaler.transform(X_test)

In [78]:
from torch.utils.data import TensorDataset,DataLoader
#Subtract 1 because target values range from 1 to 7 and we want 0 to 6
train_dataset=TensorDataset(torch.FloatTensor(X_train),torch.tensor(y_train - 1, dtype=torch.long))
train_loader=DataLoader(train_dataset,batch_size=256,shuffle=True,num_workers=2,persistent_workers=True)

In [79]:
dev_dataset=TensorDataset(torch.FloatTensor(X_dev),torch.tensor(y_dev - 1, dtype=torch.long))
dev_loader=DataLoader(dev_dataset,batch_size=256,shuffle=True,num_workers=2,persistent_workers=True)

In [140]:
test_dataset=TensorDataset(torch.FloatTensor(X_test),torch.tensor(y_test - 1, dtype=torch.long))
test_loader=DataLoader(test_dataset,batch_size=256,shuffle=True,num_workers=2,persistent_workers=True)

In [81]:
def evaluate(model,data_loader,metric,device):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch,y_batch in data_loader:
            X_batch,y_batch=X_batch.to(device),y_batch.to(device)
            y_pred=model(X_batch)
            metric.update(y_pred,y_batch)
    return metric.compute()
        

In [82]:
import time
import os
import torch

def train(model, optimizer, loss_fn, metric, train_loader,
          n_epochs,device, track_valid=True,valid_loader=None, early_stopping=False, patience=10,
          checkpoint_path=None, scheduler=None, mode="max"):
    if track_valid and valid_loader is None:
        raise ValueError("valid_loader is None with track_valid=True")
    if early_stopping:
        track_valid = True
        
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    
    best_metric = -float('inf') if mode == "max" else float('inf')
    patience_counter = 0
    
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        t0 = time.time()
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            metric.update(y_pred, y_batch)
            
        t1 = time.time()
        train_metric = metric.compute().item()
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        
        summary = (f"Epoch {epoch + 1}/{n_epochs}, "
                   f"train loss: {history['train_losses'][-1]:.4f}, "
                   f"train metric: {history['train_metrics'][-1]:.4f}, ")
        
        if track_valid:
            valid_metric = evaluate(model, valid_loader, metric,device).item()
            history["valid_metrics"].append(valid_metric)
            
            is_best = (valid_metric > best_metric) if mode == "max" else (valid_metric < best_metric)
            
            if is_best:
                torch.save(model.state_dict(), checkpoint_path)
                best_metric = valid_metric
                best_tag = " (best)"
                patience_counter = 0
            else:
                best_tag = ""
                patience_counter += 1
                
            summary += f"valid metric: {history['valid_metrics'][-1]:.4f}{best_tag}"
            
        print(summary + f" in {t1 - t0:.1f}s")
        
        if scheduler is not None:
            scheduler.step()
                
        if early_stopping and patience_counter >= patience:
            print("Early stopping triggered")
            break
    if track_valid and os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        
    return history

In [83]:
#only pin+nonblockingCPU times: total: 48.5 s Wall time: 43.5 s
#nothing CPU times: total: 47.9 s Wall time: 43.1 s
#persist workers+numworkers=4 CPU times: total: 32.9 s Wall time: 29.5 s
#persist workers +numworkes=8(max) CPU times: total: 32.8 s Wall time: 30.2 s
#persist workers +numworkes=2 CPU times: total: 31.4 s Wall time: 28.4 s BEST!!!!!
#numworkes=2CPU times: total: 35.4 s Wall time: 37 s
#persist workers +numworkes=2+pin memoryCPU times: total: 35 s Wall time: 29.1 s
#persist workers +numworkes=2+pin memory+nonblocking CPU times: total: 34.5 s Wall time: 28.8 s

In [109]:
import torch.nn as nn
class MLP(nn.Module):
    def __init__(self,n_featurs,n_hidden,n_outputs):
        super().__init__()
        self.mlp=nn.Sequential(
            nn.Linear(n_featurs,n_hidden),
            nn.ReLU(),
            #nn.Dropout(0.2),
            nn.Linear(n_hidden,n_hidden),
            nn.ReLU(),
            #nn.Dropout(0.2),
            nn.Linear(n_hidden,n_hidden),
            nn.ReLU(),
            #nn.Dropout(0.2),
            nn.Linear(n_hidden,n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden,n_outputs)
        )
    def forward(self,X):
        return self.mlp(X)

In [110]:
def he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

In [111]:
import torchmetrics
metric=torchmetrics.Accuracy(task='multiclass',num_classes=n_classes).to(device)

In [112]:
n_epochs=10

In [113]:
torch.manual_seed(42)
xentropy=nn.CrossEntropyLoss()
model=MLP(n_features,196,n_classes).to(device)
optimizer=torch.optim.Adam(model.parameters())
model.apply(he_init)

MLP(
  (mlp): Sequential(
    (0): Linear(in_features=54, out_features=196, bias=True)
    (1): ReLU()
    (2): Linear(in_features=196, out_features=196, bias=True)
    (3): ReLU()
    (4): Linear(in_features=196, out_features=196, bias=True)
    (5): ReLU()
    (6): Linear(in_features=196, out_features=196, bias=True)
    (7): ReLU()
    (8): Linear(in_features=196, out_features=7, bias=True)
  )
)

In [114]:
scheduler=torch.optim.lr_scheduler.OneCycleLR(optimizer,epochs=n_epochs,steps_per_epoch=len(train_loader),max_lr=1e-1)

In [115]:
history=train(model,optimizer,xentropy,metric,train_loader,n_epochs,device,track_valid=False,scheduler=scheduler)

Epoch 1/10, train loss: 0.4746, train metric: 0.7992,  in 8.0s
Epoch 2/10, train loss: 0.3344, train metric: 0.8615,  in 8.3s
Epoch 3/10, train loss: 0.2857, train metric: 0.8834,  in 8.1s
Epoch 4/10, train loss: 0.2598, train metric: 0.8944,  in 8.1s
Epoch 5/10, train loss: 0.2406, train metric: 0.9027,  in 8.0s
Epoch 6/10, train loss: 0.2292, train metric: 0.9075,  in 8.3s
Epoch 7/10, train loss: 0.2210, train metric: 0.9116,  in 8.0s
Epoch 8/10, train loss: 0.2139, train metric: 0.9149,  in 8.4s
Epoch 9/10, train loss: 0.2083, train metric: 0.9169,  in 8.1s
Epoch 10/10, train loss: 0.2020, train metric: 0.9200,  in 8.3s


In [116]:
evaluate(model,dev_loader,metric)

tensor(0.9172, device='cuda:0')

In [117]:
import optuna

In [132]:
def train_opt(model,optimizer,criterion,train_loader):
    model.train()
    for X_batch,y_batch in train_loader:
        X_batch,y_batch=X_batch.to(device),y_batch.to(device)
        y_pred=model(X_batch)
        loss=criterion(y_pred,y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

In [133]:
pruner=optuna.pruners.MedianPruner(n_startup_trials=5,n_warmup_steps=1,interval_steps=1)

In [134]:
def objective(trial,train_loader,dev_loader):
    n_epochs=5
    n_hidden=trial.suggest_int("n_hidden",175,300)
    n_layers=trial.suggest_int("n_layers",2,4)
    layers=[nn.Linear(n_features,n_hidden),nn.ReLU()]
    for _ in range(n_layers):
        layers+=[nn.Linear(n_hidden,n_hidden),nn.ReLU()]
    layers+=[nn.Linear(n_hidden,n_classes)]
    model=nn.Sequential(*layers).to(device)
    optimizer=torch.optim.Adam(model.parameters())
    scheduler=torch.optim.lr_scheduler.OneCycleLR(optimizer,epochs=n_epochs,steps_per_epoch=len(train_loader),max_lr=1e-1)    
    for epoch in range(n_epochs):
        train_opt(model,optimizer,xentropy,train_loader)
        dev_acc=evaluate(model,dev_loader,metric).item()
        trial.report(dev_acc,epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return evaluate(model,dev_loader,metric).item()
objective_with_data=lambda trial: objective(trial,train_loader,dev_loader)

In [135]:
torch.manual_seed(42)
sampler=optuna.samplers.TPESampler(seed=42)
study=optuna.create_study(direction="maximize",sampler=sampler,pruner=pruner)

[I 2026-09-24 21:29:46,958] A new study created in memory with name: no-name-26ad8f0b-ac23-42d2-becd-1b6961b05525


In [136]:
study.optimize(objective_with_data,n_trials=10)

[I 2026-09-24 21:30:24,560] Trial 0 finished with value: 0.9025817513465881 and parameters: {'n_hidden': 222, 'n_layers': 4}. Best is trial 0 with value: 0.9025817513465881.
[I 2026-09-24 21:30:57,743] Trial 1 finished with value: 0.9051979184150696 and parameters: {'n_hidden': 267, 'n_layers': 3}. Best is trial 1 with value: 0.9051979184150696.
[I 2026-09-24 21:31:28,617] Trial 2 finished with value: 0.9069191217422485 and parameters: {'n_hidden': 194, 'n_layers': 2}. Best is trial 2 with value: 0.9069191217422485.
[I 2026-09-24 21:32:11,442] Trial 3 finished with value: 0.902925968170166 and parameters: {'n_hidden': 182, 'n_layers': 4}. Best is trial 2 with value: 0.9069191217422485.
[I 2026-09-24 21:32:52,944] Trial 4 finished with value: 0.901824414730072 and parameters: {'n_hidden': 250, 'n_layers': 4}. Best is trial 2 with value: 0.9069191217422485.
[I 2026-09-24 21:33:34,979] Trial 5 pruned. 
[I 2026-09-24 21:34:09,054] Trial 6 pruned. 
[I 2026-09-24 21:34:42,645] Trial 7 finish

In [137]:
study.best_params

{'n_hidden': 194, 'n_layers': 2}

In [138]:
torch.manual_seed(42)
n_hidden=194
n_epochs=50
layers=[nn.Linear(n_features,n_hidden),nn.ReLU()]
for _ in range(2):
    layers+=[nn.Linear(n_hidden,n_hidden),nn.ReLU()]
layers+=[nn.Linear(n_hidden,n_classes)]
model=nn.Sequential(*layers).to(device)
optimizer=torch.optim.Adam(model.parameters())
xentropy=nn.CrossEntropyLoss()
model.apply(he_init)
scheduler=torch.optim.lr_scheduler.OneCycleLR(optimizer,epochs=n_epochs,steps_per_epoch=len(train_loader),max_lr=1e-1)

In [139]:
history=train(model,optimizer,xentropy,metric,train_loader,n_epochs,device,track_valid=False,scheduler=scheduler)

Epoch 1/50, train loss: 0.4779, train metric: 0.7985,  in 7.5s
Epoch 2/50, train loss: 0.3407, train metric: 0.8606,  in 7.6s
Epoch 3/50, train loss: 0.2908, train metric: 0.8833,  in 7.7s
Epoch 4/50, train loss: 0.2626, train metric: 0.8949,  in 8.9s
Epoch 5/50, train loss: 0.2430, train metric: 0.9032,  in 8.3s
Epoch 6/50, train loss: 0.2301, train metric: 0.9086,  in 8.3s
Epoch 7/50, train loss: 0.2209, train metric: 0.9124,  in 8.3s
Epoch 8/50, train loss: 0.2155, train metric: 0.9151,  in 8.3s
Epoch 9/50, train loss: 0.2066, train metric: 0.9185,  in 8.3s
Epoch 10/50, train loss: 0.2005, train metric: 0.9213,  in 8.6s
Epoch 11/50, train loss: 0.1968, train metric: 0.9227,  in 8.2s
Epoch 12/50, train loss: 0.1917, train metric: 0.9248,  in 8.8s
Epoch 13/50, train loss: 0.1987, train metric: 0.9250,  in 7.8s
Epoch 14/50, train loss: 0.1862, train metric: 0.9274,  in 7.9s
Epoch 15/50, train loss: 0.1825, train metric: 0.9287,  in 7.8s
Epoch 16/50, train loss: 0.1795, train metric: 0.

In [141]:
evaluate(model,test_loader,metric)

tensor(0.9338, device='cuda:0')